# Offshore Wind Potential Analysis - ERA5 Data for Campos Basin

This notebook accesses ERA5 wind data from the Copernicus Climate Data Store and uses it to explore offshore wind behavior near deactivated oil and gas platforms in the Campos Basin, Brazil.

The analysis combines wind-component data, platform coordinates, interactive maps, spatial wind-speed plots, direction vectors, and a simple theoretical wind-power potential estimate.


## Notebook Roadmap

1. Define the study context and data source.
2. Set up libraries used across the notebook.
3. Retrieve ERA5 wind data through the CDS API.
4. Load and inspect the NetCDF wind dataset.
5. Map deactivated offshore platform locations.
6. Estimate wind speed at platform points.
7. Visualize regional wind intensity and direction.
8. Estimate theoretical wind-power potential for the selected platforms.


## 1. Context & Data Source

The study area is the Campos Basin, an offshore region on the southeastern Brazilian coast with existing oil and gas infrastructure. The objective is to assess whether deactivated platforms may be relevant reference points for offshore wind generation studies.

ERA5 provides gridded atmospheric reanalysis data. In this notebook, the wind components `u` and `v` are used to compute wind speed as `sqrt(u^2 + v^2)`. API credentials must remain configured locally and should not be stored in the repository.


## 2. Environment Setup

### 2.1 Libraries

All libraries used across the notebook are imported once here. This keeps the analysis cells focused on data access, transformation, and visualization.


In [ ]:
# === ERA5 API access ===
import cdsapi

# === Data manipulation ===
import numpy as np
import pandas as pd
import xarray as xr

# === Visualization ===
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap

# === Geospatial plotting ===
import cartopy.crs as ccrs
from matplotlib.lines import Line2D

# === Project data paths ===
DATA_PATH = "../data/download_3anos.nc"
INMET_PATH = "../data/inmet_2023_sao_tome_campos.csv"


## 3. ERA5 Data Access

### 3.1 Pressure-Level Monthly Means Request

This request downloads monthly averaged ERA5 wind components at a selected pressure level for the Campos Basin region. Run this cell only when a fresh API download is needed.


In [ ]:
c = cdsapi.Client()

c.retrieve(
    'reanalysis-era5-pressure-levels-monthly-means',
    {
        'format': 'netcdf',
        'variable': [
            'u_component_of_wind', 'v_component_of_wind',
        ],
        'pressure_level': '900',
        'year': [
            '2021', '2022', '2023',
        ],
        'month': [
            '01', '02', '03',
            '04', '05', '06',
            '07', '08', '09',
            '10', '11', '12',
        ],
        'time': [
            '00:00', '01:00', '02:00',
            '03:00', '04:00', '05:00',
            '06:00', '07:00', '08:00',
            '09:00', '10:00', '11:00',
            '12:00', '13:00', '14:00',
            '15:00', '16:00', '17:00',
            '18:00', '19:00', '20:00',
            '21:00', '22:00', '23:00',
        ],
        'area': [
            -27, -45, -20, -35,
        ],
        'product_type': 'monthly_averaged_reanalysis_by_hour_of_day',
    },
    'download_3anos.nc')


### 3.2 Single-Level Hourly Reanalysis Request

This request downloads hourly ERA5 single-level wind components for 2023. It writes the result to a local NetCDF file, which is later opened by the analysis cells.


In [ ]:
c = cdsapi.Client()

c.retrieve(
    'reanalysis-era5-single-levels',
    {
        'product_type': 'reanalysis',
        'format': 'netcdf',
        'variable': [
            '10m_u_component_of_wind', '10m_v_component_of_wind',
        ],
        'year': '2023',
        'month': [
            '01', '02', '03',
            '04', '05', '06',
            '07', '08', '09',
            '10', '11', '12',
        ],
        'day': [
            '01', '02', '03',
            '04', '05', '06',
            '07', '08', '09',
            '10', '11', '12',
            '13', '14', '15',
            '16', '17', '18',
            '19', '20', '21',
            '22', '23', '24',
            '25', '26', '27',
            '28', '29', '30',
            '31',
        ],
        'time': [
            '00:00', '01:00', '02:00',
            '03:00', '04:00', '05:00',
            '06:00', '07:00', '08:00',
            '09:00', '10:00', '11:00',
            '12:00', '13:00', '14:00',
            '15:00', '16:00', '17:00',
            '18:00', '19:00', '20:00',
            '21:00', '22:00', '23:00',
        ],
        'area': [
             -27, -45, -20, -35,
        ],
    },
    'download_single_level_2023.nc')


## 4. Platform-Level Wind Speed and Location Map

### 4.1 Wind Speed at Deactivated Platform Points

This cell opens the NetCDF data, defines the platform coordinates, selects the nearest ERA5 grid point for each platform, and computes wind speed from the `u` and `v` components.

### 4.2 Interactive Platform Map

The same cell also builds a Folium map with platform markers to validate the geographic distribution of the selected assets.


In [ ]:
titulo = "Marcação das Plataformas na Bacia de Campos Desativadas"

titulo_html = """<h1 align="center" style="font-family: verdana; font-size:16px; font-weight:bold; text-transform: uppercase;">{}</h1>""".format(titulo)

# Load NetCDF file data
ds = xr.open_dataset(DATA_PATH)
#, engine='netcdf4'

# Filter data to keep the time steps for the three days (March 22, 23, and 24)
#data_3_days = ds.sel(time=(ds.time.dt.day >= 22) & (ds.time.dt.day <= 24))

# Filter data for the 775 hPa pressure level
#data_3_days_775 = data_3_days.sel(level=775)

# Assume you have platform latitude and longitude data
plat_coords = {
    'NAMORADO 2 (PNA-2)'   : {'latitude': -22.45073, 'longitude': -40.41175},
    'PETROBRAS 26 (P-26)': {'latitude': -22.4684,  'longitude': -40.02869},
    'PETROBRAS 32 (P-32)': {'latitude': -22.2051,  'longitude': -40.1431},
    'PETROBRAS 37 (P-37)': {'latitude': -22.4868,  'longitude': -40.09779},
    'PETROBRAS IX'       : {'latitude': -22.57358, 'longitude': -40.82192},
    'PETROBRAS XIX'      : {'latitude': -22.3927,  'longitude': -40.05438},
    'PETROBRAS XXXIII'   : {'latitude': -22.37,    'longitude': -40.0267},
    'VERMELHO 1 (PVM-1)' : {'latitude': -22.16065, 'longitude': -40.27872},
    'VERMELHO 2 (PVM-2)' : {'latitude': -22.17535, 'longitude': -40.29147},
}

# List to store wind intensity results by platform
wind_speeds = []

# Calculate wind intensity at each platform point
for platform, coords in plat_coords.items():
    lat, lon = coords['latitude'], coords['longitude']
    
    # Use NetCDF data to obtain wind intensity at the target point
    u = ds['u'].sel(latitude=lat, longitude=lon, method='nearest')
    v = ds['v'].sel(latitude=lat, longitude=lon, method='nearest')
    
    # Compute wind intensity from u and v components
    wind_speed = (u**2 + v**2)**0.5
    
    # Add wind intensity to the list
    wind_speeds.append({'Plataforma': platform, 'Intensidade do Vento (m/s)': wind_speed.values[0]})

# Converta a lista de resultados em um DataFrame para facilitar a manipulação
df_wind_speeds = pd.DataFrame(wind_speeds)

# Display DataFrame with wind intensities by platform
print(df_wind_speeds)

# Create an interactive Folium map centered on Campos Basin
m = folium.Map(location=[-22.8, -40.3], zoom_start=8)
m.get_root().html.add_child(folium.Element(titulo_html))

# Add markers for oil and gas platform locations in the Campos Basin
platforms = {

    'NAMORADO 2 (PNA-2)' :[-22.45073, -40.41175],
    'PETROBRAS 26 (P-26)' :[-22.4684, -40.02869],
    'PETROBRAS 32 (P-32)' :[-22.2051, -40.1431],
    'PETROBRAS 37 (P-37)' :[-22.4868, -40.09779],
    'PETROBRAS IX' :[-22.57358, -40.82192],
    'PETROBRAS XIX' :[-22.3927, -40.05438],
    'PETROBRAS XXXIII' :[-22.37, -40.0267],
    'VERMELHO 1 (PVM-1)' :[-22.16065, -40.27872],
    'VERMELHO 2 (PVM-2)' :[-22.17535, -40.29147],
    
}

# Função para atribuir cores com base na intensidade do vento

def color_wind_speed(speed):
    if speed < 5.0:
        return 'green'
    elif 5.0 <= speed < 12.0:
        return 'beige'
    elif 12.0 <= speed < 25.0:
        return 'orange'
    else:
        return 'black'

# Assume you have wind intensity data for each platform
wind_speeds = {
    'NAMORADO 2 (PNA-2)'   : df_wind_speeds.iloc[0]['Intensidade do Vento (m/s)'],  # Substitua pelos valores reais das intensidades do vento
    'PETROBRAS 26 (P-26)': df_wind_speeds.iloc[1]['Intensidade do Vento (m/s)'],
    'PETROBRAS 32 (P-32)': df_wind_speeds.iloc[2]['Intensidade do Vento (m/s)'],
    'PETROBRAS 37 (P-37)': df_wind_speeds.iloc[3]['Intensidade do Vento (m/s)'],
    'PETROBRAS IX'       : df_wind_speeds.iloc[4]['Intensidade do Vento (m/s)'],
    'PETROBRAS XIX'      : df_wind_speeds.iloc[5]['Intensidade do Vento (m/s)'],
    'PETROBRAS XXXIII'   : df_wind_speeds.iloc[6]['Intensidade do Vento (m/s)'],
    'VERMELHO 1 (PVM-1)' : df_wind_speeds.iloc[7]['Intensidade do Vento (m/s)'],
    'VERMELHO 2 (PVM-2)' : df_wind_speeds.iloc[8]['Intensidade do Vento (m/s)'],
}

# Add markers to the map
for platform, (lat, lon) in platforms.items():
    # Get platform wind intensity
    wind_speed = wind_speeds.get(platform, 0.0)  # Valor padrão 0.0 se não houver dados
    
    color = color_wind_speed(wind_speed)
    
    # Add markers with colors based on wind intensity
    folium.Marker(
        location=[lat, lon],
        popup=f'Plataforma: {platform}<br>Intensidade do Vento: {wind_speed:.2f} m/s',
        icon=folium.Icon(color=color)
    ).add_to(m)

m.save('mapa_interativo.html')
# Display the interactive map
m


## 5. Regional Wind Intensity

### 5.1 Interactive Heatmap of Mean Wind Speed

This cell samples a latitude/longitude grid across the study area, computes wind speed for each point, and displays the regional intensity pattern using a Folium heatmap.


In [ ]:
#Título

titulo = "Media do Vento na Região da Bacia de Campos"

titulo_html = """<h1 align="center" style="font-family: verdana; font-size:16px; font-weight:bold; text-transform: uppercase;">{}</h1>""".format(titulo)

# Load NetCDF file data
ds = xr.open_dataset(DATA_PATH, engine='netcdf4')

# Initial map coordinates (Campos Basin center)
latitude_inicial = -22.7366
longitude_inicial = -40.1116

# Create a Folium map object
m = folium.Map(location=[latitude_inicial, longitude_inicial], zoom_start=8)
m.get_root().html.add_child(folium.Element(titulo_html))

# List to store wind intensity results by latitude and longitude
wind_speeds = []

# Set grid density
density = 0.05  # Exemplo: 0.1 graus de separação

# Create a coordinate grid
lats = np.arange(ds.latitude.min(), ds.latitude.max(), density)
lons = np.arange(ds.longitude.min(), ds.longitude.max(), density)

# Compute wind intensity for each grid point (latitude, longitude)
for lat in lats:
    for lon in lons:
        u = ds['u'].sel(latitude=lat, longitude=lon, method='nearest')
        v = ds['v'].sel(latitude=lat, longitude=lon, method='nearest')
        wind_speed = (u**2 + v**2)**0.5

        # Check for valid values before appending to the list
        if not np.isnan(wind_speed).any():
            wind_speeds.append([lat, lon, wind_speed.values[0].astype(np.float64)])  # Converta para float64

# Determine minimum and maximum wind intensity values
min_wind_speed = np.nanmin([wind_speed for _, _, wind_speed in wind_speeds])
max_wind_speed = np.nanmax([wind_speed for _, _, wind_speed in wind_speeds])

# Create a custom color palette using seaborn
cmap = sns.color_palette("coolwarm", as_cmap=True)

# Normalize wind intensity values between 0 and 1 to match the custom color palette
norm = plt.Normalize(min_wind_speed, max_wind_speed)

# Crie uma função de gradiente personalizada para mapear a intensidade do vento para uma cor na paleta de cores personalizada
def calculate_color(wind_speed):
    return cmap(norm(wind_speed))

# Map colors according to wind intensities using the custom gradient function
colors = [calculate_color(wind_speed) for _, _, wind_speed in wind_speeds]

# Create a heatmap object with data and mapped colors

heat_data = [[lat, lon, wind_speed] for lat, lon, wind_speed in wind_speeds]

HeatMap(heat_data, gradient=None, min_opacity=0.5, radius=15, blur=16, overlay=True, control=True, name='Heat Map', show=True, legend_name='Intensidade do Vento (m/s)', gradient_fill_colors=colors).add_to(m)

# Add a bold text marker with the place name
place_name = 'Bacia de Campos'
bold_place_name = '<strong>' + place_name + '</strong>'

folium.map.Marker(
    [-22.7366, -40.1116],
    icon=folium.DivIcon(
        html=f'<div>{bold_place_name}</div>',
        icon_size=(120, 36),  # Tamanho do ícone (ajuste conforme necessário)
        icon_anchor=(0, 0),
    ),
).add_to(m)

# Visualize the map in Jupyter Notebook or save it as an HTML file
m


## 6. Temporal Wind Behavior

### 6.1 Mean Wind Speed Over Time

This cell averages wind speed across latitude and longitude to show how regional wind intensity changes through time.


In [ ]:
# Load NetCDF file data
ds = xr.open_dataset(DATA_PATH)

# Compute wind intensity from u and v components
u = ds['u']
v = ds['v']
wspd = (u**2 + v**2)**0.5

# Calcule a média da intensidade do vento ao longo do ano de 2022
wspd_mean = wspd.mean(dim=['longitude', 'latitude',])

# Plote o gráfico da intensidade do vento
plt.figure(figsize=(12, 6))
wspd_mean.plot.line(marker='o', markersize=3, linestyle='-', color='b')
plt.title('Intensidade Média do Vento (2022)')
plt.xlabel('Tempo')
plt.xticks(rotation=45)
plt.ylabel('Intensidade do Vento (m/s)')
plt.grid(True)
plt.tight_layout()
plt.show()

# Close NetCDF file
ds.close()


## 7. Spatial Wind Analysis

### 7.1 Mean Wind Speed Map

This cell computes mean wind speed over time and plots the spatial distribution with Cartopy, adding platform markers for geographic reference.


In [ ]:
# Load NetCDF file data
ds = xr.open_dataset(DATA_PATH, engine='netcdf4')

# Filter data to keep the time steps for the three days (March 22, 23, and 24)
#data_3_days = ds.sel(time=(ds.time.dt.day >= 22) & (ds.time.dt.day <= 24))

# Filter data for the 775 hPa pressure level
#data_3_days_775 = data_3_days.sel(level=775)

# Compute wind speed (wspd) from u and v components
u = ds['u']
v = ds['v']
wspd = (u**2 + v**2)**0.5

# Calcula a média da velocidade do vento para cada ponto de grade ao longo dos horários e dias
wspd_mean = wspd.mean(dim=['time'])

# Configura a projeção Plate Carrée centrada no Brasil
projection = ccrs.PlateCarree(central_longitude=-45)

# Plota o gráfico da variação da média da velocidade do vento
plt.figure(figsize=(25, 26))
ax = plt.axes(projection=projection)
ax.coastlines()
im = wspd_mean.plot(ax=ax,  transform=ccrs.PlateCarree(), add_colorbar=False)

# Add color bar manually
cbar = plt.colorbar(im, orientation='horizontal', pad=0.05)
cbar.set_label('Velocidade do Vento (m/s)', fontsize=30)
cbar.ax.tick_params(labelsize=20)

# Adiciona marcadores para as localizações das plataformas de petróleo e gás
platforms = {
    'NAMORADO 2 (PNA-2)' :(-22.45073, -40.41175),
    'PETROBRAS 26 (P-26)' :(-22.4684, -40.02869),
    'PETROBRAS 32 (P-32)' :(-22.2051, -40.1431),
    'PETROBRAS 37 (P-37)' :(-22.4868, -40.09779),
    'PETROBRAS IX' :(-22.57358, -40.82192),
    'PETROBRAS XIX' :(-22.3927, -40.05438),
    'PETROBRAS XXXIII' :(-22.37, -40.0267),
    'VERMELHO 1 (PVM-1)' :(-22.16065, -40.27872),
    'VERMELHO 2 (PVM-2)' :(-22.17535, -40.29147),
}

for platform, (lat, lon) in platforms.items():
    ax.plot(lon, lat, marker='o', color='red', markersize=10, transform=ccrs.PlateCarree())

ax.gridlines(draw_labels=True, linestyle='--', color='gray')
plt.title('Média da Velocidade do Vento no Ano de 2022 - 1000 HPA', fontsize=30)
plt.xlabel('Longitude', fontsize=30)
plt.ylabel('Latitude', fontsize=30)
plt.savefig('grafico_velocidade_vento_1000HPA.png')
plt.show()

# Close the NetCDF file
ds.close()


### 7.2 Wind Speed and Direction Map

This cell combines wind-speed magnitude with direction vectors derived from the mean `u` and `v` wind components.


In [ ]:
# Load NetCDF file data
ds = xr.open_dataset(DATA_PATH, engine='netcdf4')

# Filter data to keep the time steps for the three days (March 22, 23, and 24)
#data_3_days = ds.sel(time=(ds.time.dt.day >= 22) & (ds.time.dt.day <= 24))

# Filter data for the 775 hPa pressure level
#data_3_days_775 = data_3_days.sel(level=775)

# Compute wind speed (wspd) from u and v components
u = ds['u']
v = ds['v']
wspd = (u**2 + v**2)**0.5

# Calcula a média da velocidade do vento para cada ponto de grade ao longo dos horários e dias
wspd_mean = wspd.mean(dim=['time'])

# Calcula a direção da corrente de vento
wind_direction = np.arctan2(v.mean(dim='time'), u.mean(dim='time')) * 180 / np.pi

# Configura a projeção Plate Carrée centrada no Brasil
projection = ccrs.PlateCarree(central_longitude=-45)

# Plota o gráfico da variação da média da velocidade do vento e a direção da corrente de vento
fig = plt.figure(figsize=(12, 8))
ax = plt.axes(projection=projection)
ax.coastlines()

# Plota a variação da média da velocidade do vento
wspd_mean.plot(ax=ax, transform=ccrs.PlateCarree(), add_colorbar=True,
               cbar_kwargs={'label': 'Velocidade do Vento (m/s)'})

# Plota a direção da corrente de vento usando setas
quiver = ax.quiver(wspd_mean.longitude.values, wspd_mean.latitude.values,
                   u.mean(dim='time').values, v.mean(dim='time').values,
                   angles='xy', scale_units='xy', scale=30, color='red', alpha=0.7,
                   transform=ccrs.PlateCarree(), zorder=5)

# Adiciona marcadores para as localizações das plataformas de petróleo e gás
platforms = {
    'NAMORADO 2 (PNA-2)' :(-22.45073, -40.41175),
    'PETROBRAS 26 (P-26)' :(-22.4684, -40.02869),
    'PETROBRAS 32 (P-32)' :(-22.2051, -40.1431),
    'PETROBRAS 37 (P-37)' :(-22.4868, -40.09779),
    'PETROBRAS IX' :(-22.57358, -40.82192),
    'PETROBRAS XIX' :(-22.3927, -40.05438),
    'PETROBRAS XXXIII' :(-22.37, -40.0267),
    'VERMELHO 1 (PVM-1)' :(-22.16065, -40.27872),
    'VERMELHO 2 (PVM-2)' :(-22.17535, -40.29147),
}

for platform, (lat, lon) in platforms.items():
    ax.plot(lon, lat, marker='o', color='red', markersize=8, transform=ccrs.PlateCarree())

ax.gridlines(draw_labels=True, linestyle='--', color='gray')
plt.title('Média da Velocidade e Direção da Corrente de Vento no ano de 2022 - 1000HPA')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Add a legend for wind flow direction
ax.quiverkey(quiver, X=0.05, Y=0.01, U=10,
             label='Direção da Corrente de Vento', labelpos='E')

plt.savefig('grafico_direcao_vento_1000HPA.png')
plt.show()

# Close the NetCDF file
ds.close()


## 8. Offshore Wind Potential

### 8.1 Theoretical Wind-Power Potential by Platform

This cell applies a simplified wind-power equation using assumed air density, rotor radius, swept area, and turbine efficiency. The result is a first-order theoretical estimate, not a full engineering feasibility model.


In [ ]:
# Load NetCDF file data
ds = xr.open_dataset(DATA_PATH, engine='netcdf4')

# ... Data processing code ...

# Actual values for variables
densidade_ar = 1.225  # kg/m³
raio_rotor = 50  # metros (para um aerogerador com diâmetro de 100 metros)
area_pa = 3.14159 * (raio_rotor ** 2)  # m²
eficiencia_aerogerador = 0.4  # 40%

# Calcula o potencial de geração eólica nos pontos marcados
potencial_eolico = 0.5 * densidade_ar * area_pa * wspd_mean ** 3 * eficiencia_aerogerador

# Configura a projeção Plate Carrée centrada no Brasil
projection = ccrs.PlateCarree(central_longitude=-45)

# Plota o gráfico da variação da média da velocidade do vento e a direção da corrente de vento
fig = plt.figure(figsize=(12, 8))
ax = plt.axes(projection=projection)
ax.coastlines()

# Define different colors for each platform
cores = ['blue', 'green', 'red', 'purple', 'orange', 'brown', 'pink', 'gray', 'cyan']

# Plota a variação da média da velocidade do vento
wspd_mean.plot(ax=ax, transform=ccrs.PlateCarree(), add_colorbar=True,
               cbar_kwargs={'label': 'Velocidade do Vento (m/s)'})

# Plota a direção da corrente de vento usando setas
quiver = ax.quiver(wspd_mean.longitude.values, wspd_mean.latitude.values,
                   u.mean(dim='time').values, v.mean(dim='time').values,
                   angles='xy', scale_units='xy', scale=30, color='red', alpha=0.7,
                   transform=ccrs.PlateCarree(), zorder=5)

# Adiciona marcadores para as localizações das plataformas de petróleo e gás com cores diferentes
legend_elements = []
for i, (platform, (lat, lon)) in enumerate(platforms.items()):
    ax.plot(lon, lat, marker='o', color=cores[i], markersize=8, transform=ccrs.PlateCarree())
    potencial = potencial_eolico.sel(latitude=lat, longitude=lon, method='nearest').values
    legend_elements.append(Line2D([0], [0], marker='o', color=cores[i], markersize=8, 
                                  label=f'{platform}: {potencial:.2f} W'))

# Add side legend with potential values
ax.legend(handles=legend_elements, title='Potencial', loc='lower right', bbox_to_anchor=(1, 0.1))

ax.gridlines(draw_labels=True, linestyle='--', color='gray')
plt.title('Média da Velocidade e Direção da Corrente de Vento no Ano de 2022 - 1000HPA')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Add a legend for wind flow direction
ax.quiverkey(quiver, X=0.05, Y=0.01, U=10,
             label='Direção da Corrente de Vento', labelpos='E')

plt.savefig('pootencial_mediavelocidade_direcao_vento_1000hpa.png')
plt.show()

# Close the NetCDF file
ds.close()


## 9. ERA5 vs INMET Validation Check

### 9.1 Monthly Wind-Speed Comparison - S?o Tom? Station (2023)

This validation check compares the ERA5 nearest-grid wind speed around the INMET S?o Tom? station with observed monthly mean wind speed from station `A620 - Campos dos Goytacazes - S?o Tom?`.

The purpose is not to prove perfect equivalence between offshore reanalysis data and a land-based station, but to provide an external consistency check using public meteorological observations.


In [ ]:
# Load ERA5 processed data and INMET S?o Tom? observations
import pandas as pd

inmet_raw = pd.read_csv(
    INMET_PATH,
    sep=";",
    skiprows=10,
    decimal=",",
    parse_dates=["Data Medicao"],
    dayfirst=True,
)

inmet = inmet_raw.rename(
    columns={
        "Data Medicao": "date",
        "VENTO, VELOCIDADE MEDIA MENSAL (AUT)(m/s)": "inmet_mean_wind_speed_ms",
        "VENTO, VELOCIDADE MAXIMA MENSAL (AUT)(m/s)": "inmet_max_wind_speed_ms",
    }
)
inmet["month"] = inmet["date"].dt.to_period("M")

station_lat = -22.04166666
station_lon = -41.05166666

ds = xr.open_dataset(DATA_PATH)
u_station = ds["u"].sel(latitude=station_lat, longitude=station_lon, method="nearest")
v_station = ds["v"].sel(latitude=station_lat, longitude=station_lon, method="nearest")
era5_station_wspd = (u_station**2 + v_station**2) ** 0.5

era5 = era5_station_wspd.to_dataframe(name="era5_wind_speed_ms").reset_index()
era5["month"] = era5["time"].dt.to_period("M")
era5_2023 = era5[era5["month"].dt.year == 2023][["month", "era5_wind_speed_ms"]]

validation = inmet.merge(era5_2023, on="month", how="inner")
validation["difference_ms"] = validation["era5_wind_speed_ms"] - validation["inmet_mean_wind_speed_ms"]

correlation = validation["era5_wind_speed_ms"].corr(validation["inmet_mean_wind_speed_ms"])
mae = validation["difference_ms"].abs().mean()

print(f"Correlation (ERA5 vs INMET monthly mean wind speed, 2023): {correlation:.3f}")
print(f"Mean absolute difference: {mae:.3f} m/s")
display(validation[["date", "inmet_mean_wind_speed_ms", "era5_wind_speed_ms", "difference_ms"]])

ds.close()


### 9.2 Results - Validation Check

The comparison provides a reproducible consistency check between the ERA5 nearest-grid estimate and the INMET S?o Tom? station observations for 2023.

Because S?o Tom? is a land-based station and the project focuses on offshore platform points, differences are expected. The comparison is therefore interpreted as supporting evidence for the general wind-speed behavior, not as a definitive validation of platform-level offshore estimates.


## 10. Final Discussion

The notebook provides an exploratory view of offshore wind behavior in the Campos Basin using ERA5 data. The maps and platform-level estimates help identify spatial patterns, but final project feasibility would require a longer climatological period, turbine-specific power curves, bathymetry, grid connection analysis, environmental constraints, and higher-resolution local validation data.
